[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/data-engineering-from-scratch-to-advanced/blob/main/notebooks/topics/19_cloud-security.ipynb)

*Data Engineering from Scratch to Advanced · Notebook 19 of 21*

# Cloud, Cost & Security

Running data platforms on AWS responsibly: S3, Glue, EMR, Athena and Redshift, IAM permissions, encryption, masking and pseudonymising personal data, access control, privacy law, reliability targets and controlling the bill.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** Deny by default, grant only what's needed, and treat personal data as if it were your own family's.

**What interviewers really test here:** IAM logic, handling personal data, choosing services, and investigating a surprise bill.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. The cloud data stack *(Basics)*
- [ ] 2. Permissions: least privilege *(Core)*
- [ ] 3. Personal data and privacy *(Core)*
- [ ] 4. Reliability and the bill *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import numpy as np
import matplotlib.pyplot as plt
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Security checklist

- Roles and temporary credentials, not long-lived keys.
- Least privilege, reviewed regularly.
- Encryption at rest and in transit.
- Block public access to buckets by default.
- Secrets in a manager, never in code.
- Masked or synthetic data outside production.
- Audit logs kept and monitored.
- Network isolation with private endpoints.

---
# Part 2: Lessons, step by step

## Lesson 1: The cloud data stack

*Level: Basics*

Most data platforms run in the cloud, and the concepts carry across providers. Here's a typical AWS design, which is close to the stack many Indian and global companies use:



```text
Sources → Kinesis/MSK or DMS → S3 (raw) → Glue/EMR Spark → S3 (clean, Parquet)
                                   ↓                         ↓
                             Glue Catalog ──────→ Athena / Redshift → BI
                 Orchestrated by Airflow (MWAA)     Secured by IAM
```



| Service | Use it for |
| --- | --- |
| S3 | the lake: cheap, durable storage |
| Glue Catalog | table definitions, shared by engines |
| Glue / EMR | running Spark, serverless or as clusters |
| Athena | SQL over S3, paying per scan |
| Redshift | a warehouse for heavy, repeated BI |
| Kinesis / MSK | streaming |
| Lambda | small event-driven jobs |



**Serverless or cluster?** At made-up prices of 5 per TB scanned and 4 per cluster hour, scanning 100 TB a month costs 500 against 800 for 200 cluster hours, so serverless wins. At 5,000 TB it's 25,000 against 800, so the cluster wins. Do this sum for your own workload.



**S3 basics:** files upload in parts, so a 10 GB file at 100 MB per part has 100 parts, and a failed part alone is retried. Lifecycle rules move old data to cheaper tiers, and delete it eventually.



```python
import boto3
s3 = boto3.client("s3")                  # credentials come from an IAM role
s3.upload_file("day.parquet", "my-lake", "orders/day=2026-01-05/day.parquet")
```

**✏️ Your turn:** you now have what you need for [Q18](#q18), [Q10](#q10), [Q9](#q9). Try one now, or carry on and do them in Part 3.

## Lesson 2: Permissions: least privilege

*Level: Core*

**IAM** decides who can do what, to which resource. Three rules cover most of it:



1. **Everything's denied by default.**
2. **An explicit Allow grants access.**
3. **An explicit Deny always wins.**



```python
{"Effect": "Allow", "Action": "s3:Get*", "Resource": "lake/*"}
{"Effect": "Deny",  "Action": "s3:*",    "Resource": "lake/secret/*"}
```



| Request | Result | Why |
| --- | --- | --- |
| GetObject lake/a | allow | matches the Allow |
| GetObject lake/secret/x | deny | explicit Deny wins |
| PutObject lake/a | deny | nothing allows it |



**Least privilege:** give each job only what it needs. Treat `"Action": "*"` or `"Resource": "*"` as a warning sign.



**Prefer roles to long-lived keys.** A job assumes a role and gets temporary credentials. Any stored keys must be rotated, and audited for age.



**Access at the data level:**



| Control | Example |
| --- | --- |
| Role-based | analysts read sales, not hr |
| Row-level | Asha sees only India rows |
| Column-level or masking | phone shows as \*\*\*\*\*\*3210 |



Unknown users should get nothing. **Encrypt** in transit (TLS) and at rest (KMS keys). **Audit logs** answer "who read this table?", which you'll need for an investigation.

**✏️ Your turn:** you now have what you need for [Q12](#q12), [Q1](#q1), [Q2](#q2), [Q13](#q13), [Q3](#q3), [Q6](#q6), [Q7](#q7), [Q11](#q11). Try one now, or carry on and do them in Part 3.

## Lesson 3: Personal data and privacy

*Level: Core*

Data engineers build the systems that hold people's personal information, so this is your responsibility. Laws including GDPR in Europe and India's DPDP Act set rules on it. This lesson teaches the engineering, so check real obligations with your legal team.



| Principle | What you build |
| --- | --- |
| Collect less | don't ingest columns no one needs |
| Limit purpose and access | access controls, masking |
| Keep it only as long as needed | retention rules: logs 30 days, say |
| Right to erasure | find and delete a person across tables |



**Hashing isn't anonymising.** A 10-digit phone number has only 10¹⁰ possibilities. At a million hashes a second, you can try them all in 0.12 days, and modern hardware is far faster. So a plain hash of a phone number is easily reversed.



Use a **keyed hash (HMAC)** with a secret, or a token vault. This is **pseudonymisation**: it's still personal data, since someone with the key can link it back.



**Removing names isn't enough.** A rare combination of age band and postcode can identify someone. **k-anonymity** asks for the smallest group sharing those values. A group of one means that person is unique in the data.



**Find it before it spreads.** Scan samples for columns that look like emails or phone numbers. Over half matching is a useful signal, but a scan like this misses plenty.



**Never put real personal data in dev, or in Git.** Use synthetic or masked data.

**✏️ Your turn:** you now have what you need for [Q14](#q14), [Q4](#q4), [Q15](#q15), [Q16](#q16), [Q5](#q5). Try one now, or carry on and do them in Part 3.

## Lesson 4: Reliability and the bill

*Level: Advanced*

**What the nines mean:**



| Availability | Downtime a year |
| --- | --- |
| 99% | about 3.65 days |
| 99.9% | 525.6 minutes (8.8 hours) |
| 99.99% | about 53 minutes |



**Dependencies multiply.** A chain of services at 99.9%, 99.9% and 99% gives 98.8%, worse than any single one.



**Recovery targets:** **RPO** is how much data you can lose, and **RTO** is how long you can be down. With daily backups and an 8-hour restore, targets of 6 and 4 hours both fail. **Test your restores:** a backup you've never restored isn't really a backup.



**Cost levers:**



| Lever | Example, at made-up prices |
| --- | --- |
| Spot instances | 70% off with 20% rework: 100 becomes 36 |
| But if interruptions are bad | 10% off with 50% rework: 135, which is worse |
| Storage tiers and lifecycle rules | move old data, delete what's unused |
| Parquet and partitions | far less scanned |
| Same region and AZ | avoid transfer charges |



**Transfer costs surprise people.** 500 GB to the internet at 0.09 costs 45, which is more than 1,000 GB cross-AZ at 0.01. Tag every resource by team and environment, or you can't tell who's spending what.



**Infrastructure as code** (Terraform) keeps environments reproducible and reviewable.

**✏️ Your turn:** you now have what you need for [Q19](#q19), [Q20](#q20), [Q17](#q17), [Q8](#q8). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [2]:
from collections import Counter, defaultdict


<a id="q1"></a>
### Q1. Find Overly Broad Policies

🟢 Warm-up · *Least privilege*

Return indexes of statements allowing action '\*' or resource '\*'.

**Examples**

- `statements = [('Allow', '*', 'lake/*'), ('Allow', 's3:Get*', 'lake/*'), ('Allow', 's3:Get*', '*'), ('Deny', '*', '*')]` → `[0, 2]`

In [ ]:
def broad_policies(statements):
    """
    Return indexes of statements allowing action '*' or resource '*'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(broad_policies, [
    (([('Allow', '*', 'lake/*'), ('Allow', 's3:Get*', 'lake/*'), ('Allow', 's3:Get*', '*'), ('Deny', '*', '*')],), [0, 2]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop.

```python
def broad_policies(statements):
    out = []
    for i, (effect, a, r) in enumerate(statements):
        if effect == "Allow" and (a == "*" or r == "*"):
            out.append(i)
    return out
```

**Cost:** O(s) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension.

```python
def broad_policies(statements):
    return [i for i, (e, a, r) in enumerate(statements) if e == "Allow" and "*" in (a, r)]
```

**Cost:** O(s) time, O(m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop. That costs O(s) time and O(m) space."

"I can do better: A comprehension. That's O(s) time and O(m) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/cloud-security/find-overly-broad-policies)

<a id="q2"></a>
### Q2. Role-Based Access

🟢 Warm-up · *Roles, not people*

Users have roles, and roles have tables. Return whether a user may read a table.

**Examples**

- `users = {'a': ['analyst']}, roles = {'analyst': ['sales'], 'admin': ['sales', 'hr']}, user = 'a', table = 'hr'` → `False`
- `users = {'a': ['analyst']}, roles = {'analyst': ['sales']}, user = 'a', table = 'sales'` → `True`

In [ ]:
def can_read(users, roles, user, table):
    """
    Users have roles, and roles have tables. Return whether a user may read a
    table.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(can_read, [
    (({'a': ['analyst']}, {'analyst': ['sales'], 'admin': ['sales', 'hr']}, 'a', 'hr'), False),
    (({'a': ['analyst']}, {'analyst': ['sales']}, 'a', 'sales'), True),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`any`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Nested loops.

```python
def can_read(users, roles, user, table):
    for role in users.get(user, []):
        if table in roles.get(role, []):
            return True
    return False
```

**Cost:** O(r·t) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `any`.

```python
def can_read(users, roles, user, table):
    return any(table in roles.get(r, []) for r in users.get(user, []))
```

**Cost:** O(r·t) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested loops. That costs O(r·t) time and O(1) space."

"I can do better: `any`. That's O(r·t) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/cloud-security/role-based-access)

<a id="q3"></a>
### Q3. Column Masking

🟢 Warm-up · *Hide what isn't needed*

Mask a phone number to keep only the last 4 digits for non-privileged users.

**Examples**

- `phone = '9876543210', privileged = False` → `'******3210'`
- `phone = '9876543210', privileged = True` → `'9876543210'`

In [ ]:
def mask_phone(phone, privileged):
    """
    Mask a phone number to keep only the last 4 digits for non-privileged users.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(mask_phone, [
    (('9876543210', False), '******3210'),
    (('9876543210', True), '9876543210'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

An if, with string building.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A conditional expression.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** An if, with string building.

```python
def mask_phone(phone, privileged):
    if privileged:
        return phone
    stars = ""
    for _ in phone[:-4]:
        stars += "*"
    return stars + phone[-4:]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A conditional expression.

```python
def mask_phone(phone, privileged):
    return phone if privileged else "*" * (len(phone) - 4) + phone[-4:]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: An if, with string building. That costs O(n) time and O(n) space."

"I can do better: A conditional expression. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/cloud-security/column-masking)

<a id="q4"></a>
### Q4. How Easily Can It Be Reversed?

🟢 Warm-up · *Small spaces fall fast*

Return days to try every value of a 10-digit phone number at r hashes per second, rounded to 2 decimals. This is why plain hashing doesn't protect it.

**Examples**

- `digits = 10, rate = 1000000` → `0.12`
- `digits = 10, rate = 1000` → `115.74`

In [ ]:
def brute_days(digits, rate):
    """
    Return days to try every value of a 10-digit phone number at r hashes per
    second, rounded to 2 decimals. This is why plain hashing doesn't protect it.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(brute_days, [
    ((10, 1000000), 0.12),
    ((10, 1000), 115.74),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compact.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Step by step.

```python
def brute_days(digits, rate):
    space = 10 ** digits
    seconds = space / rate
    return round(seconds / 86400, 2)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compact.

```python
def brute_days(digits, rate):
    return round(10 ** digits / rate / 86400, 2)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Step by step. That costs O(1) time and O(1) space."

"I can do better: Compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/cloud-security/how-easily-can-it-be-reversed)

<a id="q5"></a>
### Q5. Retention Policy

🟢 Warm-up · *Keep it only as long as needed*

Items are (name, class, age\_days). Limits by class: logs 30, events 365, finance 2555. Return names to delete, sorted.

**Examples**

- `items = [('a', 'logs', 31), ('b', 'logs', 2), ('c', 'events', 400), ('d', 'finance', 400)]` → `['a', 'c']`

In [ ]:
def to_delete(items):
    """
    Items are (name, class, age_days). Limits by class: logs 30, events 365,
    finance 2555. Return names to delete, sorted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(to_delete, [
    (([('a', 'logs', 31), ('b', 'logs', 2), ('c', 'events', 400), ('d', 'finance', 400)],), ['a', 'c']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Ifs.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dictionary of limits.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Ifs.

```python
def to_delete(items):
    out = []
    for name, kind, age in items:
        if (kind == "logs" and age > 30) or (kind == "events" and age > 365) or (kind == "finance" and age > 2555):
            out.append(name)
    return sorted(out)
```

**Cost:** O(n) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dictionary of limits.

```python
def to_delete(items):
    limit = {"logs": 30, "events": 365, "finance": 2555}
    return sorted(n for n, k, a in items if a > limit[k])
```

**Cost:** O(n) time, O(m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Ifs. That costs O(n) time and O(m) space."

"I can do better: A dictionary of limits. That's O(n) time and O(m) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/cloud-security/retention-policy)

<a id="q6"></a>
### Q6. Who Touched This Data?

🟢 Warm-up · *Audit logs*

From (user, table, action) records, return users who read a sensitive table, sorted, and how many times.

**Examples**

- `log = [('a', 'hr', 'read'), ('a', 'hr', 'read'), ('b', 'hr', 'write'), ('c', 'sales', 'read')], table = 'hr'` → `{'a': 2}`

In [ ]:
def who_read(log, table):
    """
    From (user, table, action) records, return users who read a sensitive table,
    sorted, and how many times.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(who_read, [
    (([('a', 'hr', 'read'), ('a', 'hr', 'read'), ('b', 'hr', 'write'), ('c', 'sales', 'read')], 'hr'), {'a': 2}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A dictionary.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`Counter`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A dictionary.

```python
def who_read(log, table):
    out = {}
    for user, t, action in log:
        if t == table and action == "read":
            out[user] = out.get(user, 0) + 1
    return dict(sorted(out.items()))
```

**Cost:** O(n) time, O(u) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `Counter`.

```python
def who_read(log, table):
    return dict(sorted(Counter(u for u, t, a in log if t == table and a == "read").items()))
```

**Cost:** O(n) time, O(u) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A dictionary. That costs O(n) time and O(u) space."

"I can do better: `Counter`. That's O(n) time and O(u) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/cloud-security/who-touched-this-data)

<a id="q7"></a>
### Q7. Rotate Old Keys

🟢 Warm-up · *Secrets expire*

Return key names older than max\_days as of today, sorted.

**Examples**

- `created = {'a': 10, 'b': 95}, today = 100, max_days = 30` → `['a']`

In [ ]:
def old_keys(created, today, max_days):
    """
    Return key names older than max_days as of today, sorted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(old_keys, [
    (({'a': 10, 'b': 95}, 100, 30), ['a']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop.

```python
def old_keys(created, today, max_days):
    out = []
    for name, day in created.items():
        if today - day > max_days:
            out.append(name)
    return sorted(out)
```

**Cost:** O(k log k) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension.

```python
def old_keys(created, today, max_days):
    return sorted(n for n, d in created.items() if today - d > max_days)
```

**Cost:** O(k log k) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop. That costs O(k log k) time and O(k) space."

"I can do better: A comprehension. That's O(k log k) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/cloud-security/rotate-old-keys)

<a id="q8"></a>
### Q8. Data Transfer Costs

🟢 Warm-up · *The hidden bill*

Given GB moved per route, and made-up prices (same AZ 0, cross-AZ 0.01, cross-region 0.02, internet 0.09), return the total rounded to 2 decimals, and the largest route.

**Examples**

- `routes = {'cross-az': 1000, 'internet': 500, 'same-az': 9000}` → `(55.0, 'internet')`

In [ ]:
def transfer_cost(routes):
    """
    Given GB moved per route, and made-up prices (same AZ 0, cross-AZ 0.01,
    cross-region 0.02, internet 0.09), return the total rounded to 2 decimals,
    and the largest route.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(transfer_cost, [
    (({'cross-az': 1000, 'internet': 500, 'same-az': 9000},), (55.0, 'internet')),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dictionary comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop.

```python
def transfer_cost(routes):
    prices = {"same-az": 0, "cross-az": 0.01, "cross-region": 0.02, "internet": 0.09}
    total, best, best_cost = 0.0, None, -1
    for route, gb in routes.items():
        cost = gb * prices[route]
        total += cost
        if cost > best_cost:
            best, best_cost = route, cost
    return (round(total, 2), best)
```

**Cost:** O(r) time, O(r) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dictionary comprehension.

```python
def transfer_cost(routes):
    prices = {"same-az": 0, "cross-az": 0.01, "cross-region": 0.02, "internet": 0.09}
    costs = {r: g * prices[r] for r, g in routes.items()}
    return (round(sum(costs.values()), 2), max(costs, key=costs.get))
```

**Cost:** O(r) time, O(r) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop. That costs O(r) time and O(r) space."

"I can do better: A dictionary comprehension. That's O(r) time and O(r) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/cloud-security/data-transfer-costs)

<a id="q9"></a>
### Q9. S3 Lifecycle Rule

🟢 Warm-up · *Move and expire automatically*

Objects are (name, age). Rule: age ≥ 30 → 'infrequent', ≥ 90 → 'archive', ≥ 365 → 'delete', otherwise 'standard'. Return {name: action}.

**Examples**

- `objects = [('a', 1), ('b', 30), ('c', 100), ('d', 400)]` → `{'a': 'standard', 'b': 'infrequent', 'c': 'archive', 'd': 'delete'}`

In [ ]:
def lifecycle(objects):
    """
    Objects are (name, age). Rule: age ≥ 30 → 'infrequent', ≥ 90 → 'archive', ≥
    365 → 'delete', otherwise 'standard'. Return {name: action}.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(lifecycle, [
    (([('a', 1), ('b', 30), ('c', 100), ('d', 400)],), {'a': 'standard', 'b': 'infrequent', 'c': 'archive', 'd': 'delete'}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Ifs.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A helper in a comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Ifs.

```python
def lifecycle(objects):
    out = {}
    for name, age in objects:
        if age >= 365:
            out[name] = "delete"
        elif age >= 90:
            out[name] = "archive"
        elif age >= 30:
            out[name] = "infrequent"
        else:
            out[name] = "standard"
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A helper in a comprehension.

```python
def lifecycle(objects):
    rule = lambda a: "delete" if a >= 365 else "archive" if a >= 90 else "infrequent" if a >= 30 else "standard"
    return {n: rule(a) for n, a in objects}
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Ifs. That costs O(n) time and O(n) space."

"I can do better: A helper in a comprehension. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/cloud-security/s3-lifecycle-rule)

<a id="q10"></a>
### Q10. Multipart Upload

🟢 Warm-up · *Large files in pieces*

Return the number of parts for a file with a part size, both in MB, and how many parts re-upload if one fails (1, as only that part is retried).

**Examples**

- `size = 10000, part = 100` → `(100, 1)`
- `size = 250, part = 100` → `(3, 1)`

In [ ]:
def multipart(size, part):
    """
    Return the number of parts for a file with a part size, both in MB, and how
    many parts re-upload if one fails (1, as only that part is retried).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(multipart, [
    ((10000, 100), (100, 1)),
    ((250, 100), (3, 1)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Divide with a remainder check.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`ceil`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Divide with a remainder check.

```python
def multipart(size, part):
    n = size // part
    if size % part:
        n += 1
    return (n, 1)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `ceil`.

```python
def multipart(size, part):
    return (math.ceil(size / part), 1)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Divide with a remainder check. That costs O(1) time and O(1) space."

"I can do better: `ceil`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/cloud-security/multipart-upload)

<a id="q11"></a>
### Q11. Tag Everything

🟢 Warm-up · *Find what has no owner*

Return resources missing any required tag, as {name: missing tags sorted}.

**Examples**

- `resources = {'bucket': {'team': 'x'}, 'cluster': {'team': 'y', 'env': 'prod'}}, required = ['team', 'env']` → `{'bucket': ['env']}`

In [ ]:
def missing_tags(resources, required):
    """
    Return resources missing any required tag, as {name: missing tags sorted}.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(missing_tags, [
    (({'bucket': {'team': 'x'}, 'cluster': {'team': 'y', 'env': 'prod'}}, ['team', 'env']), {'bucket': ['env']}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Set differences.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Loops.

```python
def missing_tags(resources, required):
    out = {}
    for name, tags in resources.items():
        missing = [t for t in required if t not in tags]
        if missing:
            out[name] = sorted(missing)
    return out
```

**Cost:** O(r·t) time, O(r) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Set differences.

```python
def missing_tags(resources, required):
    need = set(required)
    return {n: sorted(need - t.keys()) for n, t in resources.items() if need - t.keys()}
```

**Cost:** O(r·t) time, O(r) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loops. That costs O(r·t) time and O(r) space."

"I can do better: Set differences. That's O(r·t) time and O(r) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/cloud-security/tag-everything)

<a id="q12"></a>
### Q12. IAM Policy Evaluation

🟡 Interview standard · *Explicit deny wins*

Statements are (effect, action pattern, resource pattern), with \* allowed. Return 'deny' if any matching statement denies, 'allow' if any allows, otherwise 'deny' by default.

**Examples**

- `statements = [('Allow', 's3:Get*', 'lake/*'), ('Deny', 's3:*', 'lake/secret/*')], action = 's3:GetObject', resource = 'lake/a'` → `'allow'`
- `statements = [('Allow', 's3:Get*', 'lake/*'), ('Deny', 's3:*', 'lake/secret/*')], action = 's3:GetObject', resource = 'lake/secret/x'` → `'deny'`
- `statements = [('Allow', 's3:Get*', 'lake/*')], action = 's3:PutObject', resource = 'lake/a'` → `'deny'`

In [ ]:
import fnmatch

def iam_check(statements, action, resource):
    """
    Statements are (effect, action pattern, resource pattern), with * allowed.
    Return 'deny' if any matching statement denies, 'allow' if any allows,
    otherwise 'deny' by default.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(iam_check, [
    (([('Allow', 's3:Get*', 'lake/*'), ('Deny', 's3:*', 'lake/secret/*')], 's3:GetObject', 'lake/a'), 'allow'),
    (([('Allow', 's3:Get*', 'lake/*'), ('Deny', 's3:*', 'lake/secret/*')], 's3:GetObject', 'lake/secret/x'), 'deny'),
    (([('Allow', 's3:Get*', 'lake/*')], 's3:PutObject', 'lake/a'), 'deny'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop with flags.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Comprehension of matching effects.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop with flags.

```python
def iam_check(statements, action, resource):
    allowed = False
    for effect, a, r in statements:
        if fnmatch.fnmatch(action, a) and fnmatch.fnmatch(resource, r):
            if effect == "Deny":
                return "deny"
            allowed = True
    return "allow" if allowed else "deny"
```

**Cost:** O(s) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Comprehension of matching effects.

```python
def iam_check(statements, action, resource):
    effects = {e for e, a, r in statements if fnmatch.fnmatch(action, a) and fnmatch.fnmatch(resource, r)}
    return "allow" if effects == {"Allow"} else "deny"
```

**Cost:** O(s) time, O(s) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop with flags. That costs O(s) time and O(1) space."

"I can do better: Comprehension of matching effects. That's O(s) time and O(s) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/cloud-security/iam-policy-evaluation)

<a id="q13"></a>
### Q13. Row-Level Security

🟡 Interview standard · *Same table, different rows*

Filter rows to the regions a user can see. Admins see all. Return ids.

**Examples**

- `rows = [(1, 'IN'), (2, 'US'), (3, 'IN')], access = {'asha': ['IN'], 'root': ['*']}, user = 'asha'` → `[1, 3]`
- `rows = [(1, 'IN'), (2, 'US')], access = {'root': ['*']}, user = 'root'` → `[1, 2]`
- `rows = [(1, 'IN')], access = {}, user = 'x'` → `[]`

In [ ]:
def visible_rows(rows, access, user):
    """
    Filter rows to the regions a user can see. Admins see all. Return ids.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(visible_rows, [
    (([(1, 'IN'), (2, 'US'), (3, 'IN')], {'asha': ['IN'], 'root': ['*']}, 'asha'), [1, 3]),
    (([(1, 'IN'), (2, 'US')], {'root': ['*']}, 'root'), [1, 2]),
    (([(1, 'IN')], {}, 'x'), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A set and a comprehension. Unknown users see nothing.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop.

```python
def visible_rows(rows, access, user):
    regions = access.get(user, [])
    out = []
    for i, region in rows:
        if "*" in regions or region in regions:
            out.append(i)
    return out
```

**Cost:** O(n) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A set and a comprehension. Unknown users see nothing.

```python
def visible_rows(rows, access, user):
    allowed = set(access.get(user, []))
    return [i for i, r in rows if "*" in allowed or r in allowed]
```

**Cost:** O(n) time, O(m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop. That costs O(n) time and O(m) space."

"I can do better: A set and a comprehension. Unknown users see nothing. That's O(n) time and O(m) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/cloud-security/row-level-security)

<a id="q14"></a>
### Q14. Pseudonymise With a Secret

🟡 Interview standard · *Hashes alone can be reversed*

A plain hash of a phone number can be undone by hashing all possible numbers. Use a keyed hash (HMAC-SHA256) instead. Return the first 8 hex characters, and whether it differs from the plain SHA-256.

**Examples**

- `value = '9876543210', key = 'secret'` → `('fc2c6287', True)`

In [ ]:
import hashlib

def pseudonymise(value, key):
    """
    A plain hash of a phone number can be undone by hashing all possible
    numbers. Use a keyed hash (HMAC-SHA256) instead. Return the first 8 hex
    characters, and whether it differs from the plain SHA-256.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pseudonymise, [
    (('9876543210', 'secret'), ('fc2c6287', True)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build it with `hmac.new`, step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compact.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Build it with `hmac.new`, step by step.

```python
def pseudonymise(value, key):
    import hmac
    digest = hmac.new(key.encode(), value.encode(), hashlib.sha256).hexdigest()
    plain = hashlib.sha256(value.encode()).hexdigest()
    return (digest[:8], digest != plain)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compact.

```python
def pseudonymise(value, key):
    import hmac
    d = hmac.digest(key.encode(), value.encode(), "sha256").hex()
    return (d[:8], d != hashlib.sha256(value.encode()).hexdigest())
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build it with `hmac.new`, step by step. That costs O(n) time and O(1) space."

"I can do better: Compact. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/cloud-security/pseudonymise-with-a-secret)

<a id="q15"></a>
### Q15. k-Anonymity

🟡 Interview standard · *Can someone be singled out?*

Given quasi-identifiers per person (such as age band and postcode), return the smallest group size. k = 1 means someone is unique, even with names removed.

**Examples**

- `rows = [('30s', '411'), ('30s', '411'), ('40s', '110')]` → `1`

In [ ]:
def k_anonymity(rows):
    """
    Given quasi-identifiers per person (such as age band and postcode), return
    the smallest group size. k = 1 means someone is unique, even with names
    removed.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(k_anonymity, [
    (([('30s', '411'), ('30s', '411'), ('40s', '110')],), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A dictionary.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`Counter`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A dictionary.

```python
def k_anonymity(rows):
    groups = {}
    for r in rows:
        groups[r] = groups.get(r, 0) + 1
    return min(groups.values())
```

**Cost:** O(n) time, O(g) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `Counter`.

```python
def k_anonymity(rows):
    return min(Counter(rows).values())
```

**Cost:** O(n) time, O(g) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A dictionary. That costs O(n) time and O(g) space."

"I can do better: `Counter`. That's O(n) time and O(g) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/cloud-security/k-anonymity)

<a id="q16"></a>
### Q16. Find Personal Data

🟡 Interview standard · *Scan before you share*

Return columns where over half of sampled values look like an email or 10-digit phone number, sorted.

**Examples**

- `sample = {'a': ['x@y.io', 'z@y.io', '1'], 'b': ['hello', '9876543210', 'k'], 'c': ['9876543210', '9876500000']}` → `['a', 'c']`

In [ ]:
def pii_columns(sample):
    """
    Return columns where over half of sampled values look like an email or
    10-digit phone number, sorted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pii_columns, [
    (({'a': ['x@y.io', 'z@y.io', '1'], 'b': ['hello', '9876543210', 'k'], 'c': ['9876543210', '9876500000']},), ['a', 'c']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Checks in loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A regular expression.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Checks in loops.

```python
def pii_columns(sample):
    out = []
    for col, values in sample.items():
        hits = 0
        for v in values:
            if ("@" in v and "." in v) or (v.isdigit() and len(v) == 10):
                hits += 1
        if hits * 2 > len(values):
            out.append(col)
    return sorted(out)
```

**Cost:** O(c·s) time, O(c) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A regular expression.

```python
def pii_columns(sample):
    import re
    p = re.compile(r"\S+@\S+\.\S+|\d{10}$")
    return sorted(c for c, v in sample.items() if sum(bool(p.match(x)) for x in v) * 2 > len(v))
```

**Cost:** O(c·s) time, O(c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Checks in loops. That costs O(c·s) time and O(c) space."

"I can do better: A regular expression. That's O(c·s) time and O(c) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/cloud-security/find-personal-data)

<a id="q17"></a>
### Q17. Spot vs On-Demand

🟡 Interview standard · *Cheap but interruptible*

Spot saves a discount share but can be interrupted, adding a share of rework time. Return (on-demand cost, spot cost), rounded to 2 decimals, and the cheaper.

**Examples**

- `cost = 100, discount = 0.7, rework = 0.2` → `(100, 36.0, 'spot')`
- `cost = 100, discount = 0.1, rework = 0.5` → `(100, 135.0, 'on-demand')`

In [ ]:
def spot_vs_ondemand(cost, discount, rework):
    """
    Spot saves a discount share but can be interrupted, adding a share of rework
    time. Return (on-demand cost, spot cost), rounded to 2 decimals, and the
    cheaper.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(spot_vs_ondemand, [
    ((100, 0.7, 0.2), (100, 36.0, 'spot')),
    ((100, 0.1, 0.5), (100, 135.0, 'on-demand')),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compact.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Step by step.

```python
def spot_vs_ondemand(cost, discount, rework):
    spot = cost * (1 - discount) * (1 + rework)
    name = "spot" if spot < cost else "on-demand"
    return (round(cost, 2), round(spot, 2), name)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compact.

```python
def spot_vs_ondemand(cost, discount, rework):
    s = cost * (1 - discount) * (1 + rework)
    return (round(cost, 2), round(s, 2), "spot" if s < cost else "on-demand")
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Step by step. That costs O(1) time and O(1) space."

"I can do better: Compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/cloud-security/spot-vs-on-demand)

<a id="q18"></a>
### Q18. Athena or a Cluster?

🟡 Interview standard · *Break-even*

With a made-up serverless price per TB scanned, and a cluster's hourly price, return monthly costs for t TB scanned and h cluster hours, rounded to 2 decimals, and the cheaper.

**Examples**

- `t = 100, price_tb = 5, h = 200, price_hour = 4` → `(500, 800, 'serverless')`
- `t = 5000, price_tb = 5, h = 200, price_hour = 4` → `(25000, 800, 'cluster')`

In [ ]:
def serverless_or_cluster(t, price_tb, h, price_hour):
    """
    With a made-up serverless price per TB scanned, and a cluster's hourly
    price, return monthly costs for t TB scanned and h cluster hours, rounded to
    2 decimals, and the cheaper.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(serverless_or_cluster, [
    ((100, 5, 200, 4), (500, 800, 'serverless')),
    ((5000, 5, 200, 4), (25000, 800, 'cluster')),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Multiply and compare.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compact.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Multiply and compare.

```python
def serverless_or_cluster(t, price_tb, h, price_hour):
    a = t * price_tb
    b = h * price_hour
    if a < b:
        return (round(a, 2), round(b, 2), "serverless")
    return (round(a, 2), round(b, 2), "cluster")
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compact.

```python
def serverless_or_cluster(t, price_tb, h, price_hour):
    a, b = t * price_tb, h * price_hour
    return (round(a, 2), round(b, 2), "serverless" if a < b else "cluster")
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Multiply and compare. That costs O(1) time and O(1) space."

"I can do better: Compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/cloud-security/athena-or-a-cluster)

<a id="q19"></a>
### Q19. Availability Math

🟡 Interview standard · *What do the nines mean?*

Return allowed downtime minutes a year for a target, and the availability of services that all must work, rounded to 2 and 4 decimals.

**Examples**

- `target = 0.999, services = [0.999, 0.999, 0.99]` → `(525.6, 0.988)`

In [ ]:
def availability(target, services):
    """
    Return allowed downtime minutes a year for a target, and the availability of
    services that all must work, rounded to 2 and 4 decimals.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(availability, [
    ((0.999, [0.999, 0.999, 0.99]), (525.6, 0.988)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Multiply step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`math.prod`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Multiply step by step.

```python
def availability(target, services):
    minutes = 365 * 24 * 60 * (1 - target)
    chain = 1.0
    for s in services:
        chain *= s
    return (round(minutes, 2), round(chain, 4))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `math.prod`.

```python
def availability(target, services):
    return (round(525600 * (1 - target), 2), round(math.prod(services), 4))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Multiply step by step. That costs O(n) time and O(1) space."

"I can do better: `math.prod`. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/cloud-security/availability-math)

<a id="q20"></a>
### Q20. Recovery Objectives

🟡 Interview standard · *RPO and RTO*

With backups every b hours and restore taking r hours: worst-case data loss (RPO) is b and downtime (RTO) is r. Return whether both meet targets, and which fail.

**Examples**

- `b = 4, r = 2, rpo = 6, rto = 4` → `(True, [])`
- `b = 24, r = 8, rpo = 6, rto = 4` → `(False, ['rpo', 'rto'])`

In [ ]:
def recovery(b, r, rpo, rto):
    """
    With backups every b hours and restore taking r hours: worst-case data loss
    (RPO) is b and downtime (RTO) is r. Return whether both meet targets, and
    which fail.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(recovery, [
    ((4, 2, 6, 4), (True, [])),
    ((24, 8, 6, 4), (False, ['rpo', 'rto'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two ifs.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Two ifs.

```python
def recovery(b, r, rpo, rto):
    failed = []
    if b > rpo:
        failed.append("rpo")
    if r > rto:
        failed.append("rto")
    return (not failed, failed)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension.

```python
def recovery(b, r, rpo, rto):
    failed = [n for n, ok in (("rpo", b <= rpo), ("rto", r <= rto)) if not ok]
    return (not failed, failed)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two ifs. That costs O(1) time and O(1) space."

"I can do better: A comprehension. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/cloud-security/recovery-objectives)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. How does IAM evaluate a request?</b></summary>

Denied by default. An explicit Allow permits it, and an explicit Deny always overrides.

</details>

<details>
<summary><b>2. Anonymisation vs pseudonymisation?</b></summary>

Anonymised data can't reasonably be linked back to a person. Pseudonymised data can, with a key or lookup, so it's still personal data.

</details>

<details>
<summary><b>3. Why is hashing a phone number weak?</b></summary>

The space of numbers is small, so an attacker can hash them all. Use a keyed hash or tokens.

</details>

<details>
<summary><b>4. How do you implement the right to erasure?</b></summary>

Use lineage to find all copies, delete by user key across tables, expire snapshots and old files, handle backups by policy, and log what was done.

</details>

<details>
<summary><b>5. EMR vs Glue vs Athena?</b></summary>

Athena is for serverless SQL over S3. Glue runs serverless Spark jobs with little management. EMR gives more control and can be cheaper at steady scale.

</details>

<details>
<summary><b>6. What are RPO and RTO?</b></summary>

RPO is the maximum acceptable data loss. RTO is the maximum acceptable downtime.

</details>

<details>
<summary><b>7. Scenario: an S3 bucket with customer data was public. What do you do?</b></summary>

Close access at once, and tell security and legal. Use access logs to see what was read, rotate anything exposed, and add a guardrail that blocks public buckets.

</details>

<details>
<summary><b>8. Scenario: your AWS bill spiked overnight. How do you investigate?</b></summary>

Use cost tools by service and tag. Look for forgotten clusters, cross-region transfer, or runaway jobs. Set budgets and alerts so that you hear sooner next time.

</details>

<details>
<summary><b>9. Scenario: a job fails with AccessDenied. How do you debug it?</b></summary>

Find which role it runs as, the exact action and resource from the error, then look for missing allows, explicit denies, bucket policies and KMS key permissions.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: The cloud data stack
- [ ] I can explain: Permissions: least privilege
- [ ] I can explain: Personal data and privacy
- [ ] I can explain: Reliability and the bill

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Find Personal Data
- [ ] [Q17](#q17) Spot vs On-Demand
- [ ] [Q18](#q18) Athena or a Cluster?
- [ ] [Q19](#q19) Availability Math
- [ ] [Q20](#q20) Recovery Objectives

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.